# Effect sizes between diagnoses

Marianna Georgiou, Department of Psychiatry, Psychotherapy and Psychosomatics, Faculty of Medicine, RWTH Aachen, Aachen, Germany
**Email:** mariannag43@gmail.com
**Publication:**  Latent subtypes of longitudinal postpartum trajectories for identifying healthy women at risk for affective conditions

Compare participant-averaged raw trajectories between HC, AD, and PPD, then plot Cohen’s d intervals alongside score distributions.

## Input

`files/interm_files/full_subscales_trajectories_withoutnorm_{coh}.xlsx`. Default `coh = [1, 2]`; the alternative is `[3]`. Discovery requires CODE_x and EPDS totals ending in `_y`; validation uses CODE and unsuffixed EPDS totals.

## Outputs

- `files/tables/{cohort}_raw_cohens.xlsx`: effect sizes and bootstrap intervals for seven measures.
- `files/plots/{cohort}_raw_cohens_violin.png`, `.pdf`, and `.tiff`: six-measure figure (EPDS total is omitted from the figure).

`{cohort}` is `disc` or `val`; `{coh}` is the cohort list. 

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.patheffects as path_effects

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2
from pathlib import Path

BASE_DIR = Path.cwd()
INPUT_PATH = BASE_DIR / "files" / "inputs"
OUTPUT_FILES = BASE_DIR / "files" / "interm_files"
OUTPUT_PLOTS = BASE_DIR / "files" / "plots"
OUTPUT_TABLES = BASE_DIR / "files" / "tables"
for directory in (OUTPUT_FILES, OUTPUT_PLOTS, OUTPUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def set_global_plot_style():

    """Apply the figure typography and Seaborn style."""
    sns.set_context("paper", font_scale=1.2)

    plt.rcParams['font.family'] = 'Arial'
    plt.rc('font', size=9)
    plt.rc('axes', titlesize=9)
    plt.rc('axes', labelsize=9)
    plt.rc('xtick', labelsize=9)
    plt.rc('ytick', labelsize=9)
    plt.rc('legend', fontsize=9)
    plt.rc('figure', titlesize=9)

set_global_plot_style()

np.set_printoptions(precision=3, suppress=True)

colors = {
    ('HC', 0): '#0072B2',
    ('HC', 1): '#003F5C',
    ('HC', 2): '#56B4E9',
    ('AD', 0): '#D55E00',
    ('AD', 1): '#B35806',
    ('AD', 2): '#E69F00',
    ('PPD', 0): '#009E73',
    ('PPD', 1): '#00441B',
    ('PPD', 2): '#66C2A5',
}

FIGSIZE_2X1 = (14, 6)
FIGSIZE_2X2 = (14, 12)
FIGSIZE_1X1 = (7, 6)

custom_palette = {
    'HC': colors[('HC', 0)],
    'AD': colors[('AD', 0)],
    'PPD': colors[('PPD', 0)]
}


In [ ]:
coh = [1, 2]


## 2. Prepare raw trajectories

Recode ND → HC and AS → AD, reshape scores to long format, and set MPAS T0 to missing because it was not measured.

In [ ]:
input_raw_disc = OUTPUT_FILES / f'full_subscales_trajectories_withoutnorm_{coh}.xlsx'
raw_disc = pd.read_excel(input_raw_disc)
print(type(raw_disc))
if coh == [1, 2]:
    raw_disc.rename(columns={'CODE_x': 'Diagnosis'}, inplace=True)
    raw_disc['Diagnosis'].replace({'ND': 'HC', 'AS': 'AD'}, inplace=True)
    raw_disc["MPAS T0"] = np.nan
elif coh == [3]:
    raw_disc.rename(columns={'CODE': 'Diagnosis'}, inplace=True)
    raw_disc['Diagnosis'].replace({'ND': 'HC', 'AS': 'AD'}, inplace=True)
    raw_disc["MPAS T0"] = np.nan

raw_disc.columns


In [ ]:
stress_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Stress T0', 'Stress T1', 'Stress T2', 'Stress T3', 'Stress T4'],
    var_name='Time',
    value_name='Stress'
)

mood_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Mood T0', 'Mood T1', 'Mood T2', 'Mood T3', 'Mood T4'],
    var_name='Time',
    value_name='Mood'
)

mpas_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['MPAS T0', 'MPAS T1', 'MPAS T2', 'MPAS T3', 'MPAS T4'],
    var_name='Time',
    value_name='MPAS'
)

anhedonia_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Anhedonie_t0', 'Anhedonie_t1', 'Anhedonie_t2', 'Anhedonie_t3', 'Anhedonie_t4'],
    var_name='Time',
    value_name='Anhedonia'
)

anxiety_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Anxiety_t0', 'Anxiety_t1', 'Anxiety_t2', 'Anxiety_t3', 'Anxiety_t4'],
    var_name='Time',
    value_name='Anxiety'
)

depression_df = raw_disc.melt(
    id_vars=['ID', 'Diagnosis'],
    value_vars=['Depression_t0', 'Depression_t1', 'Depression_t2', 'Depression_t3', 'Depression_t4'],
    var_name='Time',
    value_name='Depression'
)
if coh==[3]:
    epds_df = raw_disc.melt(
        id_vars=['ID', 'Diagnosis'],
        value_vars=['EPDStotal_t0', 'EPDStotal_t1', 'EPDStotal_t2', 'EPDStotal_t3', 'EPDStotal_t4'],
        var_name='Time',
        value_name='EPDS'
    )
else:
    epds_df = raw_disc.melt(
        id_vars=['ID', 'Diagnosis'],
        value_vars=['EPDStotal_t0_y', 'EPDStotal_t1_y', 'EPDStotal_t2_y', 'EPDStotal_t3_y', 'EPDStotal_t4_y'],
        var_name='Time',
        value_name='EPDS'
    )
epds_df


In [ ]:
desired_pairs = [('HC', 'AD'), ('HC', 'PPD'), ('AD', 'PPD')]

for g1, g2 in desired_pairs:
    print(g1, g2)


## 3. Participant-level effect sizes

Average available time points within each participant and diagnosis before comparing HC–AD, HC–PPD, and AD–PPD. MPAS therefore averages T1–T4; other measures include T0–T4. Each participant contributes one average per measure.

Cohen’s d is `(mean of group 2 − mean of group 1) / pooled sample SD`. Independently resample observations within each group 10,000 times, resetting a NumPy generator to seed 42 for each call, and use the 2.5th–97.5th percentiles for the interval. Empty groups, insufficient nonmissing data, and zero pooled variance are not fully guarded against.

In [ ]:
from itertools import combinations

def cohen_d_bootstrap(group1, group2, n_bootstrap=10000, alpha=0.05, random_state=42):

    """Return group-2 minus group-1 Cohen’s d and a percentile bootstrap interval."""
    rng = np.random.default_rng(random_state)

    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    pooled_std = np.sqrt(((len(group1) - 1)*std1**2 + (len(group2) - 1)*std2**2) / (len(group1) + len(group2) - 2))
    # Positive values indicate a higher mean in group 2.
    mean_diff = mean2 - mean1
    cohen_d = mean_diff / pooled_std

    boot_d = []
    for _ in range(n_bootstrap):
        sample1 = rng.choice(group1, size=len(group1), replace=True)
        sample2 = rng.choice(group2, size=len(group2), replace=True)
        m1, m2 = np.mean(sample1), np.mean(sample2)
        s1, s2 = np.std(sample1, ddof=1), np.std(sample2, ddof=1)
        pooled = np.sqrt(((len(sample1) - 1)*s1**2 + (len(sample2) - 1)*s2**2) / (len(sample1) + len(sample2) - 2))
        d = (m2 - m1) / pooled
        boot_d.append(d)

    ci_lower = np.percentile(boot_d, 100 * alpha / 2)
    ci_upper = np.percentile(boot_d, 100 * (1 - alpha / 2))

    return cohen_d, ci_lower, ci_upper

def average_per_participant(df, value_col):

    """Average available time points within participant and group."""
    return df.groupby(['ID', 'Diagnosis'])[value_col].mean().reset_index()

def compare_diagnoses(df_avg, value_col):
    """Calculate effect sizes for the fixed group comparisons."""
    print(value_col)
    diagnoses = df_avg['Diagnosis'].unique()
    print(diagnoses)
    desired_pairs = [('HC', 'AD'), ('HC', 'PPD'), ('AD', 'PPD')]
    results = []

    for g1, g2 in desired_pairs:
        print(g1,g2)
        group1 = df_avg[df_avg['Diagnosis'] == g1][value_col].values
        group2 = df_avg[df_avg['Diagnosis'] == g2][value_col].values

        cohen_d, ci_lower, ci_upper = cohen_d_bootstrap(group1, group2)

        results.append({
            'Group1': g1,
            'Group2': g2,
            'Cohens_d': cohen_d,
            'CI_lower': ci_lower,
            'CI_upper': ci_upper
        })

    return pd.DataFrame(results)

stress_avg = average_per_participant(stress_df, 'Stress')
stress_results = compare_diagnoses(stress_avg, 'Stress')
stress_results['Symptom'] = 'Stress'
print(stress_results)

mood_avg = average_per_participant(mood_df, 'Mood')
mood_results = compare_diagnoses(mood_avg, 'Mood')
mood_results['Symptom'] = 'Mood'
print(mood_results)

anh_avg = average_per_participant(anhedonia_df, 'Anhedonia')
anh_results = compare_diagnoses(anh_avg, 'Anhedonia')
anh_results['Symptom'] = 'Anhedonia'
print(anh_results)

dep_avg = average_per_participant(depression_df, 'Depression')
dep_results = compare_diagnoses(dep_avg, 'Depression')
dep_results['Symptom'] = 'Depression'
print(dep_results)

anx_avg = average_per_participant(anxiety_df, 'Anxiety')
anx_results = compare_diagnoses(anx_avg, 'Anxiety')
anx_results['Symptom'] = 'Anxiety'
print(anx_results)

mpas_avg = average_per_participant(mpas_df, 'MPAS')
mpas_results = compare_diagnoses(mpas_avg, 'MPAS')
mpas_results['Symptom'] = 'MPAS'
print(mpas_results)

epds_avg = average_per_participant(epds_df, 'EPDS')
epds_results = compare_diagnoses(epds_avg, 'EPDS')
epds_results['Symptom'] = 'EPDS'
print(epds_results)

epds_results


## 4. Export and visualize

Export all seven measures, then display six measures as effect-size forest plots and violin plots of participant averages. Positive d indicates a higher group-2 mean; its clinical direction depends on the measure. No multiplicity adjustment is applied to these bootstrap intervals.

In [ ]:
effect_size_df = pd.concat([
    stress_results,
    mood_results,
    anh_results,
    dep_results,
    anx_results,
    mpas_results,
    epds_results
], ignore_index=True)

effect_size_df

if coh == [3]:
    effect_size_df.to_excel(OUTPUT_TABLES / "val_raw_cohens.xlsx", index=False)
elif coh == [1, 2]:
    effect_size_df.to_excel(OUTPUT_TABLES / "disc_raw_cohens.xlsx", index=False)

effect_size_df.head()


In [ ]:
effect_size_df['Comparison'] = effect_size_df['Group1'] + ' vs ' + effect_size_df['Group2']

effect_size_df = effect_size_df[[
    'Symptom', 'Comparison', 'Group1', 'Group2', 'Cohens_d', 'CI_lower', 'CI_upper'
]]

stress_avg['Symptom'] = 'Stress'
mood_avg['Symptom'] = 'Mood'
anh_avg['Symptom'] = 'Anhedonia'
dep_avg['Symptom'] = 'Depression'
anx_avg['Symptom'] = 'Anxiety'
mpas_avg['Symptom'] = 'MPAS'
epds_avg['Symptom'] = 'EPDS'

stress_avg = stress_avg.rename(columns={'Stress': 'Value'})
mood_avg = mood_avg.rename(columns={'Mood': 'Value'})
anh_avg = anh_avg.rename(columns={'Anhedonia': 'Value'})
dep_avg = dep_avg.rename(columns={'Depression': 'Value'})
anx_avg = anx_avg.rename(columns={'Anxiety': 'Value'})
mpas_avg = mpas_avg.rename(columns={'MPAS': 'Value'})
epds_avg = epds_avg.rename(columns={'EPDS': 'Value'})

long_avg_df = pd.concat([
    stress_avg,
    mood_avg,
    anh_avg,
    dep_avg,
    anx_avg,
    mpas_avg,
    epds_avg
], ignore_index=True)

effect_size_df


In [ ]:
symptom_list = ['Stress', 'Mood', 'MPAS', 'Anhedonia', 'Anxiety', 'Depression']
n_symptoms = len(symptom_list)
sns.set_style("ticks")

custom_palette = {
    'HC': '#0072B2',
    'AD': '#D55E00',
    'PPD': '#009E73'
}

fig = plt.figure(figsize=(6, 2 * n_symptoms))
gs = fig.add_gridspec(n_symptoms, 2, width_ratios=[1.1, 1.5], wspace=0.4, hspace=0.9)

effect_size_df['Symptom'] = pd.Categorical(effect_size_df['Symptom'], categories=symptom_list, ordered=True)

max_comparisons = effect_size_df.groupby('Symptom').size().max()
shared_y_pos = np.arange(max_comparisons)

for i, symptom in enumerate(symptom_list):
    symptom_data = effect_size_df[effect_size_df['Symptom'] == symptom].copy()
    symptom_data = symptom_data.sort_values(by='Comparison').reset_index(drop=True)

    n_rows = len(symptom_data)
    y_pos = shared_y_pos[:n_rows]
    cohens_d = symptom_data['Cohens_d']
    ci_lowers = cohens_d - symptom_data['CI_lower']
    ci_uppers = symptom_data['CI_upper'] - cohens_d

    if symptom == 'Stress':
        row_title = 'Stress self-assessment'
    elif symptom == 'Mood':
        row_title = 'Mood self-assessment'
    elif symptom in ['Anhedonia', 'Anxiety', 'Depression']:
        row_title = f"{symptom} (EPDS)"
    else:
        row_title = symptom

    ax_left = fig.add_subplot(gs[i, 0])
    ax_left.errorbar(cohens_d, y_pos, xerr=[ci_lowers, ci_uppers], fmt='o',  capsize=4, color='black')
    ax_left.axvline(0, linestyle='--', color='black', lw=1)
    ax_left.set_yticks(y_pos)
    ax_left.set_yticklabels(symptom_data['Comparison'])
    ax_left.invert_yaxis()
    ax_left.set_xlabel("Cohen's d (95% CI)")
    ax_left.set_xlim(-5, 5)
    ax_left.set_ylim(max_comparisons - 0.5, -0.5)
    ax_left.grid(True, axis='x', color='lightgrey', linestyle='--', linewidth=0.7)
    sns.despine(ax=ax_left, left=True, bottom=False)

    ax_left.set_title(row_title,   pad=20, ha='center')

    ax_right = fig.add_subplot(gs[i, 1])
    sub = long_avg_df[long_avg_df['Symptom'] == symptom]

    sns.violinplot(
        data=sub,
        x='Diagnosis',
        y='Value',
        palette=custom_palette,
        ax=ax_right,
        inner='box',
        linewidth=1.2
    )

    ax_right.set_ylabel(f"{symptom} Score")
    ax_right.set_xlabel('')
    ax_right.tick_params(axis='x')
    ax_right.tick_params(axis='y')
    ax_right.grid(True, axis='y', linestyle='--', color='lightgrey', linewidth=0.7)
    ax_right.grid(False, axis='x')

plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

if coh == [3]:
    aux_name = 'val'
elif coh == [1, 2]:
    aux_name = 'disc'

for ext in ['png', 'pdf', 'tiff']:
        save_path = OUTPUT_PLOTS / f"{aux_name}_raw_cohens_violin.{ext}"
        fig.savefig(save_path, dpi=600, bbox_inches='tight', format=ext)
